# Generalized CPD (GCPD) — OCT→MRI alignment on the I58 dataset

Companion to the **CPD** and **SYMNet** notebooks, for
[*Coherent Point Drift Revisited for Non-Rigid Shape Matching and Registration*](https://github.com/AoxiangFan/GeneralizedCoherentPointDrift)
(Fan, Ma, Tian, Mei & Liu, **CVPR 2022**) — "GCPD".

GCPD is a **mesh / surface** method, not a volumetric or point-cloud one. Its deformation is
`T = Y + U·C`, where **U is the Laplace–Beltrami (LB) eigenbasis of the source mesh** — that intrinsic
spectral basis *is* the paper's "generalized (non-Euclidean) kernel". This notebook is a faithful
**Python port** of the official MATLAB core (`GCPD.m`, `GCPD_initial.m`, `figtreeRecoveryDistribution.m`,
`norm2.m`) plus the SHOT/functional-map initialization (`MWP2.m`), validated against the repo's own
cat meshes before use.

**Same as the CPD notebook:** Drive mount + identical data paths, the low-memory NIfTI loader, Otsu
specimen masks, **Dice** and **surface-residual (mm)**, and the red-contour multi-planar overlays.

**Different (because GCPD is a spectral mesh method):**
- The two shapes are obtained by **marching cubes** on the MRI and OCT specimen masks, then cleaned and
  decimated to triangle meshes (OCT = *source* Y, MRI = *target* X).
- Initialization follows the paper: **descriptor (SHOT if available, else WKS) + functional-map matching**.
- `figtree` (fast Gauss transform) is replaced by an **exact chunked Gauss transform** (identical result).
- Affine-only columns are replaced by **deformation-field metrics**: mean/max vertex deformation (mm),
  LB regularizer energy `trace(CᵀKC)`, final `σ²`, and inlier count.
- Variants sweep the GCPD **λ** (regularization) and **outlier** ratio — mirroring the CPD `STARTS × W_LIST` grid.

> ⚠️ Cross-modality note: GCPD's SHOT + functional-map init assumes near-isometric surfaces of similar
> topology. OCT↔MRI specimen surfaces differ in FOV/cropping, so the spectral matching may be imperfect;
> the `Dice init → Dice (warped)` columns make the actual gain explicit. GPU is not required.


In [ ]:
!pip -q install libigl trimesh scikit-image SimpleITK nibabel scipy pandas matplotlib
# SHOT descriptors are optional; fall back to WKS (Wave Kernel Signature) if pyshot is unavailable.
try:
    import pyshot; HAVE_SHOT = True
except Exception:
    !pip -q install pyshot 2>/dev/null
    try:
        import pyshot; HAVE_SHOT = True
    except Exception:
        HAVE_SHOT = False
import igl, trimesh, skimage, numpy as np
print('libigl/trimesh/skimage OK | SHOT available:', HAVE_SHOT, '| falling back to WKS' if not HAVE_SHOT else '')
print('ok')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/DATA_DIR'     # shortcut to the shared folder
!ls -la "$DATA_DIR/Datasets" 

In [ ]:
# ---------------- CONFIG ----------------
# Data paths — identical to the CPD notebook
MRI_PATH      = f'{DATA_DIR}/Datasets/I58_brainstem_MRI_cropped_to_OCT.nii.gz'
OCT_PATH      = f'{DATA_DIR}/Datasets/I58_Brainstem_mus_Slice_full_20um_corr.nii.gz'
MRI_MASK_PATH = None            # None -> Otsu + largest component
OCT_MASK_PATH = None
PRELOAD_MM     = 0.25           # block-average factor while loading (20 um OCT cannot fit in RAM)
TARGET_SPACING = 0.5            # mm, grid used for Dice, surface residual and overlays

# Mesh / GCPD params
DECIMATE_FACES = 6000           # target face count for each decimated surface mesh
MESH_SMOOTH_IT = 5              # Taubin smoothing iterations on the raw marching-cubes surface
N_EIGS         = 120            # LB eigenfunctions used as the deformation basis U  (must be < #vertices)
K0_FMAP        = 60             # spectral size for the functional-map initialization
WKS_N, WKS_VAR = 100, 6.0       # Wave Kernel Signature descriptor settings
FMAP_ITERS     = 5              # MWP-style functional-map refinement iterations
# GCPD core (repo defaults: lambda=10, outliers=0.1, MaxIter=30, tol=1e-5)
GCPD_MAXIT, GCPD_TOL = 30, 1e-5
INIT_MAXIT     = 20

# Variant grid (mirrors CPD's STARTS x W_LIST)
LAMBDA_LIST    = [1.0, 10.0, 100.0]   # GCPD regularization weight
OUTLIERS_LIST  = [0.1]                 # GCPD outlier ratio

# status thresholds
DICE_GAIN_OK = 0.02
DICE_LOW     = 0.30

OUT = '/content/results_gcpd'
import os; os.makedirs(OUT, exist_ok=True)
import numpy as np; np.random.seed(0)

In [ ]:
import os, time, numpy as np, SimpleITK as sitk, pandas as pd
from scipy import ndimage as ndi
import gzip, nibabel as nib, gc
def load_lowmem(path, preload_mm, slab_mem_mb=300):
    r = sitk.ImageFileReader(); r.SetFileName(path); r.ReadImageInformation()
    size, spacing, origin, direction = r.GetSize(), np.array(r.GetSpacing()), np.array(r.GetOrigin()), r.GetDirection()
    if len(size) < 3: raise ValueError(f'{path}: expected a 3D volume, got {len(size)}D {size}')
    size = np.array(size[:3])
    f = np.maximum(1, np.floor(preload_mm / spacing + 1e-6)).astype(int)
    print(f'{os.path.basename(path)}: size {tuple(size)}, spacing {np.round(spacing,4)} mm, {np.prod(size)/1e6:.0f}M voxels -> block factor {f}')
    px = nib.load(path).dataobj
    dtype = px.dtype; off = int(px.offset); slope, inter = px.slope, px.inter
    slope = 1.0 if slope is None or not np.isfinite(slope) or slope == 0 else float(slope); inter = 0.0 if inter is None or not np.isfinite(inter) else float(inter)
    nx_, ny_, nz_ = [int(v) for v in size]; ox, oy, oz = nx_ // f[0], ny_ // f[1], nz_ // f[2]
    nx, ny = ox * f[0], oy * f[1]
    out = np.zeros((ox, oy, oz), np.float32)
    per_z = nx_ * ny_ * dtype.itemsize
    kb = max(1, int(slab_mem_mb * 1e6 // (per_z * 2)) // int(f[2]))
    opener = gzip.open if str(path).endswith('.gz') else open
    t0 = time.time(); k0 = 0
    with opener(path, 'rb') as fh:
        fh.seek(off)
        while k0 < oz:
            k = min(kb, oz - k0); zs = k * int(f[2])
            buf = fh.read(per_z * zs)
            slab = np.frombuffer(buf, dtype=dtype, count=nx_ * ny_ * zs).reshape((nx_, ny_, zs), order='F')[:nx, :ny]
            slab = slab.astype(np.float32) if (slope == 1.0 and inter == 0.0) else slab.astype(np.float32) * slope + inter
            np.nan_to_num(slab, copy=False)
            out[:, :, k0:k0 + k] = slab.reshape(ox, f[0], oy, f[1], k, f[2]).mean(axis=(1, 3, 5))
            del slab, buf; k0 += k
            el = time.time() - t0; print(f'  {100*k0/oz:5.1f}%  elapsed {el/60:5.1f} min  ETA {el*(oz-k0)/max(k0,1)/60:5.1f} min', end='\r')
    print()
    img = sitk.GetImageFromArray(np.transpose(out, (2, 1, 0)))
    D = np.array(direction).reshape(3, 3)
    img.SetSpacing(tuple(spacing * f)); img.SetDirection(direction)
    img.SetOrigin(tuple(origin + D @ ((f - 1) / 2.0 * spacing)))
    return img

def otsu_mask(img):
    m = sitk.OtsuThreshold(sitk.Cast(img, sitk.sitkFloat32), 0, 1)
    m = sitk.BinaryMorphologicalClosing(m, [3]*3)
    cc = sitk.RelabelComponent(sitk.ConnectedComponent(m)); m = cc == 1
    return sitk.BinaryFillhole(m)
rdm = lambda p: sitk.Cast(sitk.ReadImage(p, sitk.sitkFloat32) > 0, sitk.sitkUInt8)

t0 = time.time()
mri, oct_ = load_lowmem(MRI_PATH, PRELOAD_MM), load_lowmem(OCT_PATH, PRELOAD_MM)
print('load time %.1f min' % ((time.time() - t0) / 60))
mri_m = rdm(MRI_MASK_PATH) if MRI_MASK_PATH else otsu_mask(mri)
oct_m = rdm(OCT_MASK_PATH) if OCT_MASK_PATH else otsu_mask(oct_)
print('MRI', mri.GetSize(), mri.GetSpacing(), '| OCT', oct_.GetSize(), oct_.GetSpacing())
print('mask voxels  MRI', int(sitk.GetArrayFromImage(mri_m).sum()), ' OCT', int(sitk.GetArrayFromImage(oct_m).sum()))

In [ ]:
# ======================================================================
# GCPD core — Python port of the official MATLAB repo, validated on the
# repo's cat meshes (mean surface distance 0.43 -> 0.06).
# ======================================================================
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import eigsh
from scipy.spatial import cKDTree
from scipy.spatial.distance import cdist
import igl


# ----------------------------------------------------------------------- geometry / basis
def normalize_pair(X, Y):
    """norm2.m: zero-mean, scale BOTH by X's RMS scale."""
    X = np.asarray(X, float); Y = np.asarray(Y, float)
    xm, ym = X.mean(0), Y.mean(0)
    X = X - xm; Y = Y - ym
    xscale = np.sqrt((X ** 2).sum() / X.shape[0])
    return X / xscale, Y / xscale, dict(xm=xm, ym=ym, xscale=xscale)


def lb_basis(V, F, k):
    """Laplace-Beltrami eigenbasis: smallest-k eigenpairs of (-L) phi = e M phi."""
    V = np.asfortranarray(V, dtype=np.float64); F = np.asfortranarray(F, dtype=np.int32)
    L = igl.cotmatrix(V, F)            # negative semidefinite (nv x nv)
    M = igl.massmatrix(V, F, igl.MASSMATRIX_TYPE_VORONOI)
    k = int(min(k, V.shape[0] - 2))
    evals, evecs = eigsh(-L, k=k, M=M, sigma=1e-8, which='LM')
    idx = np.argsort(evals); evals = evals[idx]; evecs = evecs[:, idx]
    evals[0] = max(evals[0], 0.0)
    return evecs, evals, M


def wks(evecs, evals, n_e=100, variance=6.0):
    """Wave Kernel Signature descriptor (functional-map initialization feature)."""
    evals = np.maximum(evals, 1e-6)
    log_e = np.log(evals)
    e_min, e_max = log_e[1], log_e[-1]
    es = np.linspace(e_min, e_max, n_e)
    sigma = (es[1] - es[0]) * variance
    phi2 = evecs ** 2
    desc = np.zeros((evecs.shape[0], n_e))
    for i, ei in enumerate(es):
        coef = np.exp(-(ei - log_e) ** 2 / (2 * sigma ** 2))
        desc[:, i] = phi2 @ coef / (coef.sum() + 1e-12)
    return desc


# ----------------------------------------------------------------------- functional-map init (MWP2.m)
def fmap_init(evecs1, evals1, M1, evecs2, evals2, M2, desc1, desc2, k0=100, num_iters=5):
    """Spectral functional-map correspondence src(1)->tgt(2); returns T12 (per-src-vertex tgt index)."""
    k0 = int(min(k0, evecs1.shape[1], evecs2.shape[1]))
    P1, P2 = evecs1[:, :k0], evecs2[:, :k0]
    A = P1.T @ (M1 @ desc1)            # (k0 x p)
    B = P2.T @ (M2 @ desc2)            # (k0 x p)
    # initial C: min_C ||C A - B|| with ridge
    C = B @ A.T @ np.linalg.inv(A @ A.T + 1e-3 * np.eye(k0))
    T = cKDTree(P2 @ C.T).query(P1)[1]                       # src -> tgt
    # MWP-style refinement: C from current map, map from C
    for _ in range(num_iters):
        C = np.linalg.lstsq(P1, P2[T], rcond=None)[0]        # C_fmap = Phi1 \ Phi2(T,:)
        T = cKDTree(P2 @ C.T).query(P1)[1]
    return T, C


# ----------------------------------------------------------------------- GCPD_initial.m
def gcpd_initial(Xdata, Ymodel, U, e, lam=10.0, gamma=0.9, theta=0.75, a=0.05,
                 max_iter=20, ecr=1e-5, minP=1e-5):
    """Outlier-robust spectral-basis registration (index-aligned Xdata/Ymodel). Returns C, inlier index."""
    K = np.diag(e); N, D = Xdata.shape; k = U.shape[1]
    C = np.zeros((k, 3)); T = Ymodel.copy()
    sigma2 = ((Xdata - Ymodel) ** 2).sum() / (N * D)
    E = 1.0; tecr = 1.0; it = 0
    while it < max_iter and abs(tecr) > ecr and sigma2 > 1e-8:
        E_old = E
        d2 = ((Xdata - T) ** 2).sum(1)
        temp1 = np.exp(-d2 / (2 * sigma2))
        temp2 = (2 * np.pi * sigma2) ** (D / 2) * (1 - gamma) / (gamma * a)
        P1 = temp1 / (temp1 + temp2)
        E = (P1 * d2).sum() / (2 * sigma2) + P1.sum() * np.log(sigma2) * D / 2 \
            - np.log(gamma) * P1.sum() - np.log(1 - gamma) * (1 - P1).sum()
        P1 = np.maximum(P1, minP)
        E = E + lam / 2 * np.trace(C.T @ K @ C)
        tecr = (E - E_old) / E
        dP = P1[:, None]
        C = np.linalg.solve(U.T @ (dP * U) + lam * sigma2 * K,
                            U.T @ (dP * Xdata) - U.T @ (dP * Ymodel))
        T = Ymodel + U @ C
        V = Xdata - T
        sigma2 = (P1 * (V ** 2).sum(1)).sum() / (D * P1.sum())
        gamma = np.clip((P1 > theta).sum() / N, 0.05, 0.95)
        it += 1
    index = np.where(P1 > theta)[0]
    return C, index


# ----------------------------------------------------------------------- Gauss transform (figtree replacement)
def gauss_transform(sources, weights, queries, sigma2, chunk=2000):
    """sum_s weights_s * exp(-||q - s||^2 / (2 sigma2)); exact, chunked over queries."""
    weights = np.atleast_2d(weights.T).T if weights.ndim == 1 else weights
    out = np.zeros((queries.shape[0], weights.shape[1]))
    for i in range(0, queries.shape[0], chunk):
        q = queries[i:i + chunk]
        Kq = np.exp(-cdist(q, sources, 'sqeuclidean') / (2 * sigma2))
        out[i:i + chunk] = Kq @ weights
    return out


# ----------------------------------------------------------------------- GCPD.m (full refine)
def gcpd(Xtarget, Ysource, U, e, C, lam=10.0, outliers=0.1, max_iter=30, tol=1e-5, sigma2=1e-4):
    """Generalized CPD EM. X=target verts, Y=source verts, U=source LB evecs. Returns deformed source T0, C."""
    K = np.diag(e); N, D = Xtarget.shape; Mm = Ysource.shape[0]
    T = Ysource + U @ C
    it = 0; ntol = tol + 10; L = 0.0
    while it < max_iter and ntol > tol and sigma2 > 10 * np.finfo(float).eps:
        L_old = L
        # E-step (figtreeRecoveryDistribution)
        outlier_tmp = (outliers * Mm * (2 * sigma2 * np.pi) ** (D / 2)) / ((1 - outliers) * N)
        sp_ = gauss_transform(T, np.ones((Mm, 1)), Xtarget, sigma2)[:, 0] + outlier_tmp   # (N,)
        Pt1 = 1 - outlier_tmp / sp_
        P1 = gauss_transform(Xtarget, (1.0 / sp_)[:, None], T, sigma2)[:, 0]               # (M,)
        PX = gauss_transform(Xtarget, Xtarget / sp_[:, None], T, sigma2)                   # (M,3)
        L = -np.log(sp_).sum() + D * N * np.log(sigma2) / 2
        ntol = abs((L - L_old) / L)
        dP = P1[:, None]
        # M-step
        C = np.linalg.solve(U.T @ (dP * U) + lam * sigma2 * K, U.T @ PX - U.T @ (dP * Ysource))
        T = Ysource + U @ C
        Np = P1.sum()
        sigma2 = abs(((Xtarget ** 2 * Pt1[:, None]).sum() + (T ** 2 * P1[:, None]).sum()
                      - 2 * (PX * T).sum()) / (Np * D))
        it += 1
    return T, C, dict(iters=it, sigma2=float(sigma2), P1=P1, Pt1=Pt1)

print('GCPD core defined.')

In [ ]:
import trimesh
# ---------- geometry helpers (physical mm <-> voxel index), shared with CPD ----------
def geom(img): return np.array(img.GetSpacing()), np.array(img.GetDirection()).reshape(3, 3), np.array(img.GetOrigin())
def idx_to_phys(img, idx): sp, D, o = geom(img); return o + (D @ (np.asarray(idx, float) * sp).T).T
def phys_to_idx(img, P):   sp, D, o = geom(img); return (np.linalg.inv(D) @ (np.asarray(P) - o).T).T / sp

# shared working grid (MRI geometry at TARGET_SPACING) for Dice / overlays
sp_ = [TARGET_SPACING] * 3
size_ = [int(round(s * o / t)) for s, o, t in zip(mri.GetSize(), mri.GetSpacing(), sp_)]
grid = sitk.Image(size_, sitk.sitkFloat32); grid.SetSpacing(sp_); grid.SetOrigin(mri.GetOrigin()); grid.SetDirection(mri.GetDirection())
mri_mg = sitk.Resample(mri_m, grid, sitk.Transform(), sitk.sitkNearestNeighbor, 0)

from skimage import measure
def mask_to_mesh(mask_img, target_faces, smooth_it=MESH_SMOOTH_IT):
    a = sitk.GetArrayFromImage(mask_img).astype(np.float32)                    # (z,y,x)
    a = ndi.binary_fill_holes(a > 0.5).astype(np.float32)
    verts_zyx, faces, _, _ = measure.marching_cubes(a, level=0.5)              # index coords (z,y,x)
    verts_phys = idx_to_phys(mask_img, verts_zyx[:, ::-1])                     # (x,y,z) index -> physical mm
    m = trimesh.Trimesh(vertices=verts_phys, faces=faces, process=True)
    trimesh.repair.fix_normals(m)
    if smooth_it: trimesh.smoothing.filter_taubin(m, iterations=smooth_it)
    Vv = np.asfortranarray(m.vertices, dtype=np.float64); Ff = np.asfortranarray(m.faces, dtype=np.int32)
    V, Fd, _, _ = igl.decimate(Vv, Ff, int(target_faces))
    # keep largest connected component for a clean 2-manifold
    md = trimesh.Trimesh(V, Fd, process=True)
    cc = md.split(only_watertight=False)
    if len(cc): md = max(cc, key=lambda c: len(c.vertices))
    print(f'  {os.path.basename(str(mask_img))} mesh: {len(md.vertices)} verts, {len(md.faces)} faces')
    return np.asarray(md.vertices), np.asarray(md.faces)

print('building meshes (OCT = source, MRI = target)...')
Vs, Fs = mask_to_mesh(oct_m, DECIMATE_FACES)        # source  (to be deformed)
Vt, Ft = mask_to_mesh(mri_m, DECIMATE_FACES)        # target
print('source(OCT)', Vs.shape, Fs.shape, '| target(MRI)', Vt.shape, Ft.shape)

In [ ]:
# ---------- normalize (norm2), LB bases, descriptors, functional-map initialization ----------
Xt_n, Ys_n, NRM = normalize_pair(Vt, Vs)            # X=target(MRI) data, Y=source(OCT) model; both /xscale
print('LB eigenbasis (source + target)... k =', N_EIGS)
U_s, e_s, M_s = lb_basis(Ys_n, Fs, N_EIGS)          # source basis U drives the deformation T = Y + U C
U_t, e_t, M_t = lb_basis(Xt_n, Ft, N_EIGS)

def shot_or_wks(V, F, evecs, evals):
    if HAVE_SHOT:
        try:
            import pyshot
            return pyshot.get_descriptors(V.astype(np.float64), F.astype(np.int64),
                                          radius=0.1 * np.ptp(V, 0).max(),
                                          local_rf_radius=0.1 * np.ptp(V, 0).max(),
                                          min_neighbors=4, n_bins=10)
        except Exception as ex:
            print('  SHOT failed, using WKS:', ex)
    return wks(evecs, evals, WKS_N, WKS_VAR)

d_s = shot_or_wks(Ys_n, Fs, U_s, e_s)
d_t = shot_or_wks(Xt_n, Ft, U_t, e_t)
print('functional-map init (descriptor + MWP-style refinement)...')
T12_0, C_fmap = fmap_init(U_s, e_s, M_s, U_t, e_t, M_t, d_s, d_t, k0=K0_FMAP, num_iters=FMAP_ITERS)
print('  initial correspondence T12_0:', T12_0.shape)

In [ ]:
from scipy.spatial import cKDTree
# ---------- metrics (Dice + surface residual shared with CPD; rest are deformation metrics) ----------
def mesh_to_gridmask(V_phys, F):
    # Voxelize a mesh (physical mm) onto the working grid -> binary array (z,y,x)
    mesh = trimesh.Trimesh(V_phys, F, process=False)
    try:
        vox = mesh.voxelized(pitch=TARGET_SPACING).fill()
        pts = vox.points                                           # occupied voxel centres (mm)
    except Exception:
        pts = V_phys
    idx = np.round(phys_to_idx(grid, pts)).astype(int)             # -> (x,y,z) voxel indices
    shape_xyz = np.array(grid.GetSize())
    keep = np.all((idx >= 0) & (idx < shape_xyz), axis=1)
    idx = idx[keep]
    out = np.zeros(shape_xyz[::-1], bool)                          # (z,y,x)
    out[idx[:, 2], idx[:, 1], idx[:, 0]] = True
    return ndi.binary_fill_holes(out)

def dice_np(a, b): a = a > 0.5; b = b > 0.5; return float(2 * (a & b).sum() / max(a.sum() + b.sum(), 1))
def surf_resid_mm(Va, Vb):
    d1 = cKDTree(Vb).query(Va)[0].mean(); d2 = cKDTree(Va).query(Vb)[0].mean()
    return float((d1 + d2) / 2)

def denorm(V_norm): return V_norm * NRM['xscale'] + NRM['xm']      # normalized -> physical MRI mm
mri_mask_arr = sitk.GetArrayFromImage(mri_mg) > 0
print('metric helpers ready.')

In [ ]:
import time
def run_variant(lam, outliers):
    t_start = time.time()
    # GCPD initial (outlier-robust), then full GCPD refine — repo pipeline
    Xdata = Xt_n[T12_0]                                             # target verts matched to each source vert
    C_init, inl = gcpd_initial(Xdata, Ys_n, U_s, e_s, lam=lam, max_iter=INIT_MAXIT)
    T0_n, C, info = gcpd(Xt_n, Ys_n, U_s, e_s, C_init, lam=lam, outliers=outliers,
                         max_iter=GCPD_MAXIT, tol=GCPD_TOL)
    secs = time.time() - t_start

    # physical-space meshes: init (norm2 placement) vs GCPD-warped source
    V_init  = denorm(Ys_n)
    V_warp  = denorm(T0_n)
    dice_init  = dice_np(mesh_to_gridmask(V_init, Fs), mri_mask_arr)
    dice_final = dice_np(mesh_to_gridmask(V_warp, Fs), mri_mask_arr)
    resid_init  = surf_resid_mm(V_init, Vt)
    resid_final = surf_resid_mm(V_warp, Vt)
    # deformation metrics (mm): per-vertex displacement U C, LB energy trace(C'KC)
    defo_mm = np.linalg.norm((U_s @ C), axis=1) * NRM['xscale']
    lb_energy = float(np.trace(C.T @ np.diag(e_s) @ C))

    if not np.isfinite(dice_final) or dice_final < 1e-6: status = 'failed (degenerate)'
    elif dice_final >= dice_init + DICE_GAIN_OK and dice_final >= DICE_LOW: status = 'succeeded (improved over init)'
    elif dice_final < dice_init - DICE_GAIN_OK: status = 'degraded vs init'
    elif dice_final < DICE_LOW: status = 'low overlap'
    else: status = 'near init (little deformable gain)'

    row = {'Tool': 'GCPD (ported)', 'lambda': lam, 'outliers': outliers, 'Status': status,
           'Dice init': dice_init, 'Dice (warped OCT vs MRI)': dice_final,
           'Surf resid init (mm)': resid_init, 'Surface residual (mm)': resid_final,
           'Mean deform (mm)': float(defo_mm.mean()), 'Max deform (mm)': float(defo_mm.max()),
           'LB energy tr(CKC)': lb_energy, 'Inliers (init)': int(len(inl)),
           'sigma2': info['sigma2'], 'GCPD iters': info['iters'], 'Time (s)': secs}
    extra = dict(V_warp=V_warp, V_init=V_init, C=C, defo_mm=defo_mm, dice_final=dice_final)
    return row, extra
print('run_variant() ready.')

In [ ]:
rows, store = [], {}
for lam in LAMBDA_LIST:
    for outliers in OUTLIERS_LIST:
        key = f'lam{lam}|out{outliers}'
        try:
            r_, ex_ = run_variant(lam, outliers); rows.append(r_); store[key] = ex_
        except Exception as e:
            rows.append({'Tool': 'GCPD', 'lambda': lam, 'outliers': outliers, 'Status': f'failed: {type(e).__name__}: {e}'})
            print('FAILED', key, e)
        last = rows[-1]
        print(f"{key:18s} {last.get('Status')}  | Dice {last.get('Dice init', float('nan')):.3f}->{last.get('Dice (warped OCT vs MRI)', float('nan')):.3f}"
              f"  resid {last.get('Surface residual (mm)', float('nan')):.2f}mm  {last.get('Time (s)', 0):.1f}s")

res = pd.DataFrame(rows)
res.to_csv(f'{OUT}/gcpd_results.csv', index=False)
pd.set_option('display.width', 260); pd.set_option('display.max_columns', 30); pd.set_option('display.max_colwidth', 60)
res.round(3)

In [ ]:
import matplotlib, matplotlib.pyplot as plt
# Overlay style identical to CPD: warped OCT specimen (gray) + MRI boundary (red) on the working grid.
mb = mri_mask_arr
D_ = np.array(grid.GetDirection()).reshape(3, 3)
ax_of = lambda phys_axis: 2 - int(np.argmax(np.abs(D_[phys_axis])))
PLANES = {'axial': ax_of(2), 'coronal': ax_of(1), 'sagittal': ax_of(0)}
sup_sgn = int(np.sign(D_[2, int(np.argmax(np.abs(D_[2])))]))
def slice_idx(plane, where='mid'):
    ax = PLANES[plane]; c = np.argwhere(mb)[:, ax]
    if where == 'mid': return int(np.median(c))
    top = int(c.max() if (plane == 'axial' and sup_sgn > 0) or plane != 'axial' else c.min())
    return int(np.clip(top - sup_sgn * round(2 / grid.GetSpacing()[0]), 0, mb.shape[ax] - 1))
def panel(a, w, plane, sl, title):
    take = lambda v: np.take(v, sl, axis=PLANES[plane])
    a.imshow(take(w.astype(float)), cmap='gray', origin='lower', interpolation='nearest')
    a.contour(take(mb.astype(float)), [0.5], colors='red', linewidths=1.0)
    a.set_title(title, fontsize=7); a.axis('off')

items = [('INIT (OCT, norm2 placement)', mesh_to_gridmask(store[list(store)[0]]['V_init'], Fs))]
for key in store: items.append((f'GCPD: {key}', mesh_to_gridmask(store[key]['V_warp'], Fs)))

ncol = 4; nrow = int(np.ceil(len(items) / ncol))
for tag, plane, where in [('axial_mid','axial','mid'), ('axial_superior','axial','sup'), ('sagittal_mid','sagittal','mid'), ('coronal_mid','coronal','mid')]:
    sl = slice_idx(plane, where); fig, axs = plt.subplots(nrow, ncol, figsize=(3.2*ncol, 4*nrow)); axs = np.atleast_2d(axs)
    for a in axs.ravel(): a.axis('off')
    for a, (name, vol) in zip(axs.ravel(), items): panel(a, vol, plane, sl, name)
    plt.suptitle(f'{tag}: warped OCT specimen (gray), MRI boundary (red)'); plt.tight_layout(); plt.savefig(f'{OUT}/overlay_contact_{tag}.png', dpi=140); plt.show()

for name, vol in items:
    fig, axs = plt.subplots(1, 4, figsize=(14, 5))
    for a, (plane, where) in zip(axs, [('axial','mid'), ('axial','sup'), ('sagittal','mid'), ('coronal','mid')]):
        panel(a, vol, plane, slice_idx(plane, where), f'{name}\n{plane} {where}')
    plt.tight_layout(); plt.savefig(f"{OUT}/overlay_{name.replace(':','').replace('|','_').replace(' ','_')}.png", dpi=140); plt.close(fig)
print('overlays saved to', OUT)

In [ ]:
# 3D view: target (MRI) surface, source (OCT) init, and GCPD-warped source; coloured by deformation magnitude.
best_key = max(store, key=lambda k: store[k]['dice_final'])
ex = store[best_key]
fig = plt.figure(figsize=(14, 5))
def scat(ax, V, c, title, cmap='viridis'):
    s = ax.scatter(V[:, 0], V[:, 1], V[:, 2], c=c, s=2, cmap=cmap)
    ax.set_title(title, fontsize=9); ax.set_box_aspect([np.ptp(V[:,0]), np.ptp(V[:,1]), np.ptp(V[:,2])]); ax.axis('off'); return s
ax1 = fig.add_subplot(1, 3, 1, projection='3d'); scat(ax1, Vt, 'lightgray', 'Target (MRI) surface')
ax2 = fig.add_subplot(1, 3, 2, projection='3d'); scat(ax2, ex['V_init'], 'tab:blue', 'Source (OCT) init')
ax3 = fig.add_subplot(1, 3, 3, projection='3d'); s = scat(ax3, ex['V_warp'], ex['defo_mm'], f'GCPD warped — {best_key}', 'magma')
fig.colorbar(s, ax=ax3, fraction=0.03, label='deformation (mm)')
plt.tight_layout(); plt.savefig(f'{OUT}/mesh3d_{best_key.replace("|","_")}.png', dpi=140); plt.show()

In [ ]:
from IPython.display import Image, display
best_idx = res['Dice (warped OCT vs MRI)'].idxmax(); best_row = res.loc[best_idx]
best_key = max(store, key=lambda k: store[k]['dice_final'])
print('Best GCPD configuration:')
print(f"- lambda / outliers   : {best_row['lambda']} / {best_row['outliers']}")
print(f"- Dice (init -> GCPD) : {best_row['Dice init']:.3f} -> {best_row['Dice (warped OCT vs MRI)']:.3f}")
print(f"- Surface residual    : {best_row['Surface residual (mm)']:.3f} mm")
print(f"- Mean / max deform   : {best_row['Mean deform (mm)']:.3f} / {best_row['Max deform (mm)']:.3f} mm")
print(f"- LB energy / sigma2  : {best_row['LB energy tr(CKC)']:.3f} / {best_row['sigma2']:.2e}\n")
name = f"GCPD: {best_key}"
img_path = f"{OUT}/overlay_{name.replace(':','').replace('|','_').replace(' ','_')}.png"
print('Multi-planar overlay for the best variant:')
display(Image(filename=img_path))